# Unit 2 — Trajectory Mining: Map Matching a Bus Line with an HMM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bgalon/geo-graph-learning/blob/main/unit-2-trajectory-mining/geoai-graph-unit2.ipynb)

> **© 2026 Ben Galon. All rights reserved.** Part of the **Geo-AI course (The Arena)** — provided to enrolled students for course use, **not for redistribution**.
>
> **AI-assisted materials.** This notebook was drafted with AI assistance (Claude Code) using curated course context and reviewed by the instructor. It is a learning reference, **not a guaranteed-correct key** — verify before you rely on it.

**The big idea — noise is a modeling decision, not a cleanup step.** A city bus
reports its GPS position about **once a minute**. One such trace is too sparse to
read as a route: between two pings the bus travels hundreds of meters, and many
roads could connect them. In this demo we run **one noise frame forward**: given
the road network *and* the noisy pings, infer the trajectory. The tool is **HMM
map matching** (Newson & Krumm 2009) — candidate road segments are hidden states,
the noisy pings are observations, and the matched route is the **Viterbi path**.

**What you will build:** a tunable HMM map matcher that recovers a Tel Aviv bus
line's shape from sparse GPS, and you will learn to turn its single load-bearing
knob — the **emission sigma** (how much GPS noise the model expects).

**The arc (9 steps):** raw pings → pool many runs → Kalman (honest failure) →
set up the HMM → Viterbi on one trace → pool *matched* traces (shape
crystallizes) → interpret + sweep sigma → **add heading + velocity** (fix the
wrong carriageway) → **run it backward** (infer the graph from pings) + bridge to
Unit 3. Steps 8–9 are advanced and *trimmable* if the live clock is tight.

**Prerequisites (Unit 1):** building an OSM road graph with `osmnx`, projecting
to a metric CRS, and reading a graph as nodes/edges.

**Library stack (classical only):** `leuvenmapmatching` (HMM), `osmnx` (road
graph), `geopandas`/`shapely`/`pyproj` (geometry + projection), `filterpy`
(Kalman), `tslearn` (DTW), `scikit-image` (skeletonize, for the inverse map
inference), `folium` (interactive maps). No deep learning here — the learned
frontier (L2MM, MMformer, DiffMM) is a slides-only topic.


## Setup

On **Colab** the cell below fetches `setup_colab.py` from the course repo and
installs the Unit 2 requirements. **Off Colab** (local `uv` env) it is a no-op.


In [ ]:
# On Colab: install Unit 2 deps. Locally: assumes `uv sync --extra unit-2` was run.
import sys
if "google.colab" in sys.modules:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bgalon/geo-graph-learning/main/setup_colab.py",
        "setup_colab.py",
    )
from setup_colab import setup_unit
setup_unit("unit-2")

## Smoke test — fail fast, before any teaching content

Bare-import every major library plus one trivial call each, and **pin the
version-sensitive Leuven matched-path accessor** (`path_pred_onlynodes`) so a
broken environment fails *here*, loudly, not three sections deep. On failure we
print a one-line banner pointing to `KNOWN_ISSUES.md` and re-raise cleanly.


In [ ]:
# --- Unit 2 smoke test ---------------------------------------------------
_smoke_err = None
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    import osmnx as ox
    import geopandas as gpd
    from shapely.geometry import Point, LineString
    from pyproj import Transformer
    import rtree                     # spatial index (Leuven candidate search)
    import scipy                     # numerics
    import networkx as nx

    from leuvenmapmatching.map.inmem import InMemMap
    from leuvenmapmatching.matcher.distance import DistanceMatcher

    from filterpy.kalman import KalmanFilter          # step 3
    from tslearn.metrics import dtw, dtw_path         # step 8 DTW teaser
    from skimage.morphology import skeletonize        # step 8 map-inference (inverse) mini-pipeline

    import folium
    from folium.plugins import HeatMap
    from folium.plugins import PolyLineTextPath, AntPath   # directed-trajectory arrows (steps 1, 5)
    from folium import LayerControl                   # basemap/topology toggle — viz-rich workhorse
    import mapclassify                                # choropleth binning
    import branca.colormap as bcm                     # folium colormaps/legends

    # --- trivial calls -------------------------------------------------
    _t = Transformer.from_crs("EPSG:4326", "EPSG:2039", always_xy=True)
    assert abs(_t.transform(34.78, 32.07)[0]) > 0           # geopandas/pyproj reproject
    # Tiny but COMPLETE matchable map: a 3-node chain with bidirectional edges,
    # rtree + edge index on (mirrors how we build the real map in step 4).
    _m = InMemMap("smoke", use_latlon=False, use_rtree=True, index_edges=True)
    for _n, _xy in [(0, (0.0, 0.0)), (1, (0.0, 10.0)), (2, (0.0, 20.0))]:
        _m.add_node(_n, _xy)
    for _a, _b in [(0, 1), (1, 0), (1, 2), (2, 1)]:
        _m.add_edge(_a, _b)
    _m.purge()
    _kf = KalmanFilter(dim_x=4, dim_z=2)                    # filterpy constructs
    assert dtw([0, 1, 2], [0, 1, 2]) == 0.0                 # tslearn DTW one-liner
    _sk = skeletonize(np.array([[0,0,0],[1,1,1],[0,0,0]], dtype=bool), method="lee")  # skimage 1-px skeleton
    assert _sk.dtype == bool                                # skeletonize returns a bool raster
    _fm = folium.Map(location=[32.07, 34.78], zoom_start=12); LayerControl().add_to(_fm)

    # --- PIN the version-sensitive matched-path accessor ----------------
    # Leuven 1.1.x exposes the Viterbi node sequence as `DistanceMatcher.path_pred_onlynodes`.
    # If a future version renames it, the demo's step-5/6 path extraction breaks — catch it now.
    assert hasattr(DistanceMatcher, "match"), "Leuven DistanceMatcher API changed"
    _probe = DistanceMatcher(_m, max_dist=50, max_dist_init=50, obs_noise=5,
                             non_emitting_states=False, min_prob_norm=None)
    _probe.match([(0.0, 1.0), (0.0, 9.0), (0.0, 19.0)])     # a track along the chain
    assert hasattr(_probe, "path_pred_onlynodes"), \
        "Leuven matched-path accessor `path_pred_onlynodes` is missing — see KNOWN_ISSUES.md"

    # contextily is a STATIC fallback only — guard it (network-dependent, optional).
    try:
        import contextily  # noqa: F401
        _HAVE_CTX = True
    except Exception:
        _HAVE_CTX = False
except Exception as exc:  # capture, do NOT raise here (IPython 9.x mangles chained SystemExit)
    _smoke_err = exc

if _smoke_err is not None:
    print("=" * 70)
    print("SMOKE TEST FAILED — environment is not ready. See KNOWN_ISSUES.md")
    print(f"  ({type(_smoke_err).__name__}: {_smoke_err})")
    print("=" * 70)
    raise _smoke_err from None        # clean re-raise OUTSIDE the except block

print("Smoke test passed — all Unit 2 libraries import and the Leuven accessor is pinned.")
print(f"contextily static-basemap fallback available: {_HAVE_CTX}")

## Data — fetch the bus line inline (self-contained, nothing pre-staged)

We use a small, pre-cut artifact: **all 16 days of Tel Aviv bus line `13429`**
(`recorded_at_time, lat, lon, bearing, velocity, line_ref, vehicle_ref`,
~42k rows, 0.72 MB), hosted on Google Drive and pulled with `gdown`. The fetch is
**cached** to `/content` so re-runs are instant.

> Provenance is fully reproducible: the optional cell at the very bottom of this
> notebook regenerates this artifact from the canonical raw archive. It is **not**
> on the class runtime path (it would pull 1.2 GB, 80% of it an irrelevant SQL
> dump).


In [ ]:
import os
import pandas as pd

DRIVE_ID = "1SdxoE7FUFE1sKSjLpXE1_rf1kmWJDd6u"   # public ta_line_13429.parquet (~0.72 MB)
LOCAL = "ta_line_13429.parquet"

if not os.path.exists(LOCAL):
    import gdown
    gdown.download(id=DRIVE_ID, output=LOCAL, quiet=False)

df = pd.read_parquet(LOCAL)
df["recorded_at_time"] = pd.to_datetime(df["recorded_at_time"], utc=True)
print(f"{len(df):,} pings · {df.vehicle_ref.nunique()} vehicles · "
      f"line_ref={df.line_ref.unique().tolist()}")
df.head(3)

**Auto-pick the busiest day**, then split each vehicle's pings into
**journeys** by cutting at gaps longer than 10 minutes (a single `vehicle_ref`
makes several runs of the line over a day, separated by layovers). The single
trace and all pooled views below come from this one day.


In [ ]:
df = df.sort_values(["vehicle_ref", "recorded_at_time"]).reset_index(drop=True)
df["day"] = df.recorded_at_time.dt.date

BUSIEST_DAY = df.groupby("day").size().idxmax()
day = df[df.day == BUSIEST_DAY].copy()

# Journey = contiguous run; cut where the per-vehicle time gap exceeds 10 min.
day["gap_s"] = day.groupby("vehicle_ref").recorded_at_time.diff().dt.total_seconds()
day["new_journey"] = (day.gap_s > 600) | day.gap_s.isna()
day["journey_n"] = day.groupby("vehicle_ref").new_journey.cumsum()
day["jid"] = day.vehicle_ref + "_" + day.journey_n.astype(str)

jsize = day.groupby("jid").size().sort_values(ascending=False)
SINGLE_JID = jsize.index[0]                       # busiest single journey = our headline trace
single = day[day.jid == SINGLE_JID].sort_values("recorded_at_time").reset_index(drop=True)

print(f"Busiest day: {BUSIEST_DAY}  ·  {len(day):,} pings  ·  {day.jid.nunique()} journeys")
print(f"Headline single journey: {SINGLE_JID}  ·  {len(single)} pings  "
      f"({single.recorded_at_time.min():%H:%M} → {single.recorded_at_time.max():%H:%M} UTC)")

A small folium helper used throughout: every map gets a **basemap toggle**
(OpenStreetMap, a light Carto basemap, and a **blank "topology-only"** layer so
route/road geometry reads on its own) plus a `LayerControl`.


In [ ]:
import folium
from folium import LayerControl

def base_map(center, zoom=13):
    "Folium map with three switchable basemaps incl. a blank topology-only layer."
    m = folium.Map(location=center, zoom_start=zoom, tiles=None, control_scale=True)
    folium.TileLayer("OpenStreetMap", name="OSM").add_to(m)
    folium.TileLayer("CartoDB positron", name="Carto (light)").add_to(m)
    # Blank background so only the geometry we draw is visible.
    folium.TileLayer(
        tiles="https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}.png",
        attr="(topology only)", name="Topology only (blank)",
    ).add_to(m)
    return m

DAY_CENTER = [day.lat.mean(), day.lon.mean()]
print("Map helper ready. Remember to add LayerControl(...) at the end of each map.")

## Step 1 — One vehicle, one route: raw pings

**Concept:** a single ~1-ping/min trace is a **Lagrangian** sample of a moving
object — sparse, and visually unconvincing as a route. This is the motivating
hook.

**DIRECT vocabulary:** trajectory · sampling rate / temporal resolution ·
Lagrangian frame · emission noise.

*What to notice (map):* the dots do **not** trace a road — the between-ping gaps
are the whole problem.

*A trajectory is not an unordered point cloud:* it is **ordered in time** and **directed** — the bus runs one way down the corridor. Toggle the "direction of travel (arrows)" layer to see START (green) → END (red). This direction is exactly the signal the **bearing-aware matcher** exploits in Step 8.


In [ ]:
import branca.colormap as bcm
from folium.plugins import PolyLineTextPath, AntPath

# (1) Interactive map of the single trace: time-coloured, numbered ping markers.
m1 = base_map([single.lat.mean(), single.lon.mean()], zoom=13)
tsec = (single.recorded_at_time - single.recorded_at_time.min()).dt.total_seconds()
cmap = bcm.linear.viridis.scale(tsec.min(), tsec.max()); cmap.caption = "seconds into the journey"
fg = folium.FeatureGroup(name="raw pings (single trace)")
for i, row in single.iterrows():
    folium.CircleMarker([row.lat, row.lon], radius=4, weight=1,
                        color=cmap(tsec[i]), fill=True, fill_opacity=0.9,
                        popup=f"#{i}  {row.recorded_at_time:%H:%M:%S}").add_to(fg)
fg.add_to(m1)

# DIRECTION OF TRAVEL: a trajectory is an *ordered, directed* sequence, not a
# point cloud. Draw the time-sorted pings as a thin polyline and stamp repeating
# arrowheads (PolyLineTextPath = static, Colab-trust-safe) pointing in travel
# order. AntPath adds an animated "flow" in the same direction (best-effort).
order = list(zip(single.lat.values, single.lon.values))   # already time-sorted
g_dir = folium.FeatureGroup(name="direction of travel (arrows)", show=True)
dir_line = folium.PolyLine(order, color="#222", weight=1.5, opacity=0.7)
dir_line.add_to(g_dir)
PolyLineTextPath(dir_line, "   ➤   ", repeat=True, offset=4,
                 attributes={"fill": "#cc3311", "font-weight": "bold", "font-size": "16"}).add_to(g_dir)
AntPath(order, color="#cc3311", weight=2, opacity=0.6, delay=800,
        dash_array=[10, 20], pulse_color="#222").add_to(g_dir)
folium.Marker(order[0], tooltip="START",
              icon=folium.Icon(color="green", icon="play", prefix="fa")).add_to(g_dir)
folium.Marker(order[-1], tooltip="END",
              icon=folium.Icon(color="red", icon="stop", prefix="fa")).add_to(g_dir)
g_dir.add_to(m1)

cmap.add_to(m1); LayerControl(collapsed=False).add_to(m1)
print("What to notice: the arrowheads run START (green) -> END (red). A trajectory is "
      "ORDERED in time and DIRECTED — the bus runs one way. That direction is the exact "
      "signal the bearing-aware matcher exploits in Step 8.")
m1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyproj import Transformer

# (2) inter-ping time-gap histogram + (3) inter-ping distance ECDF (meters, projected).
_tf = Transformer.from_crs("EPSG:4326", "EPSG:2039", always_xy=True)
sx, sy = _tf.transform(single.lon.values, single.lat.values)
step_d = np.hypot(np.diff(sx), np.diff(sy))                 # meters between consecutive pings
gap_s = single.recorded_at_time.diff().dt.total_seconds().dropna().values

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(gap_s, bins=30, color="#4477aa"); ax[0].axvline(60, color="k", ls="--", lw=1)
ax[0].set(title="Inter-ping time gap", xlabel="seconds", ylabel="count")
ax[0].text(62, ax[0].get_ylim()[1]*0.8, "~60 s cadence", fontsize=9)
xs = np.sort(step_d); ax[1].plot(xs, np.arange(1, len(xs)+1)/len(xs), color="#ee6677")
ax[1].set(title="Inter-ping distance (ECDF)", xlabel="meters moved between pings", ylabel="fraction")
plt.tight_layout(); plt.show()
print(f"Median time gap {np.median(gap_s):.0f}s · median distance moved {np.median(step_d):.0f} m between pings")

**You can now:** load a single bus's GPS trace and recognize that 1 ping/min
is too sparse to read as a route — the bus moves a median of *hundreds of meters*
between fixes.


## Step 2 — All vehicles on that route, pooled

**Concept:** overlay **every run** of the line that day. The corridor *teases*
but stays ambiguous at the edges — **more data ≠ alignment**.

**DIRECT vocabulary:** pooling · coverage · noise vs. ambiguity.

*What to notice:* the density heatmap suggests a spine, but no individual road is
resolved — density alone doesn't tell you *which* road a between-ping segment used.


In [ ]:
from folium.plugins import HeatMap

# (1) pooled pings: a density HEATMAP layer + a raw POINTS layer (toggle).
m2 = base_map(DAY_CENTER, zoom=12)
HeatMap(day[["lat", "lon"]].values.tolist(), radius=7, blur=9,
        name="ping density (heatmap)").add_to(m2)
pts = folium.FeatureGroup(name="raw pings (points)", show=False)
for lat, lon in day[["lat", "lon"]].values:
    folium.CircleMarker([lat, lon], radius=1.5, color="#333", fill=True,
                        fill_opacity=0.4, weight=0).add_to(pts)
pts.add_to(m2); LayerControl(collapsed=False).add_to(m2)
m2

In [ ]:
# (2) pings-per-journey histogram — run-length distribution.
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(jsize.values, bins=30, color="#228833")
ax.set(title=f"Pings per journey (busiest day, {day.jid.nunique()} journeys)",
       xlabel="pings in a journey", ylabel="number of journeys")
plt.tight_layout(); plt.show()
print(f"{(jsize >= 30).sum()} journeys have >= 30 pings — these are full runs of the line.")

**You can now:** pool many traces and articulate why density alone doesn't
resolve which road each between-ping segment used.


## Step 3 — Kalman smoothing on the one trace — *honest failure*

**Concept:** a 2D **constant-velocity Kalman filter** (`filterpy`) fuses a motion
model with the noisy measurements and cleans per-ping **jitter**. But it operates
in *free space* with no notion of the road network — so at 1 ping/min the road
**still doesn't surface**, because the dominant problem is between-ping ambiguity,
not jitter.

**DIRECT vocabulary:** Kalman filter · state-space / motion model · process vs.
measurement noise (Q/R) · jitter vs. between-ping ambiguity.

*What to notice:* the smoothed line is prettier but **floats off-road**; the
point-to-nearest-road distance shrinks a little but does **not** collapse to zero.


In [ ]:
from filterpy.kalman import KalmanFilter

# 2D constant-velocity filter. State = [x, vx, y, vy]; we observe position only.
# Work in projected meters (sx, sy from step 1) so Q/R are in meters.
def kalman_cv(xs, ys, dt=60.0, q=4.0, r=15.0):
    kf = KalmanFilter(dim_x=4, dim_z=2)
    kf.F = np.array([[1, dt, 0, 0], [0, 1, 0, 0],
                     [0, 0, 1, dt], [0, 0, 0, 1]], float)   # constant velocity
    kf.H = np.array([[1, 0, 0, 0], [0, 0, 1, 0]], float)    # observe x, y
    kf.R = np.eye(2) * r**2                                  # measurement noise (m^2)
    kf.Q = np.eye(4) * q**2                                  # process noise
    kf.x = np.array([xs[0], 0, ys[0], 0], float); kf.P = np.eye(4) * 50**2
    out = []
    for zx, zy in zip(xs, ys):
        kf.predict(); kf.update([zx, zy]); out.append((kf.x[0], kf.x[2]))
    return np.array(out)

kal = kalman_cv(sx, sy)
print(f"Kalman-smoothed {len(kal)} positions (constant-velocity, R=15 m, Q=4 m).")

In [ ]:
import osmnx as ox

# Build the corridor road graph ONCE, projected to ITM (meters). Three-tier
# acquisition so a flaky Overpass can't break class (Addition C):
#   1. local cache (corridor_2039.graphml)  ->  2. LIVE Overpass (osmnx)  ->
#   3. hosted BACKUP graph on Drive (gdown + gunzip, already EPSG:2039).
# Re-runs and later steps are instant once the local cache exists.
GRAPHML = "corridor_2039.graphml"
BACKUP_GRAPH_ID = "1P6SyrzDnTFld3YX2cNorpbS0JfiUhTrR"   # busiest-day corridor, EPSG:2039, 22.5k nodes
PAD = 0.004
W, S, E, N = day.lon.min()-PAD, day.lat.min()-PAD, day.lon.max()+PAD, day.lat.max()+PAD

def _load_backup_graph(out_path):
    "Fetch + gunzip the hosted backup graph (already projected to EPSG:2039)."
    import gdown, gzip, shutil
    gz = "backup_graph_2039.graphml.gz"
    gdown.download(id=BACKUP_GRAPH_ID, output=gz, quiet=True)
    with gzip.open(gz, "rb") as f_in, open(out_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)           # gunzip -> graphml
    return ox.load_graphml(out_path)              # already EPSG:2039 — no reprojection

if os.path.exists(GRAPHML):
    Gp = ox.load_graphml(GRAPHML)
else:
    try:                                          # (2) LIVE Overpass first — keeps the demo honest
        G = ox.graph_from_bbox((W, S, E, N), network_type="drive")
        Gp = ox.project_graph(G, to_crs="EPSG:2039")   # metric CRS -> sigma in METERS
        ox.save_graphml(Gp, GRAPHML)
    except Exception as exc:                      # (3) Overpass down -> hosted backup, with a banner
        print("=" * 70)
        print("OVERPASS DOWNLOAD FAILED — falling back to the hosted backup graph.")
        print(f"  ({type(exc).__name__}: {exc})")
        print("  Fetching + gunzipping backup_graph_2039.graphml.gz from Drive...")
        print("=" * 70)
        Gp = _load_backup_graph(GRAPHML)
        ox.save_graphml(Gp, GRAPHML)              # cache the backup so re-runs are instant
        print("Backup graph loaded (EPSG:2039) — class can proceed. See KNOWN_ISSUES.md.")

edges_gdf = ox.graph_to_gdfs(Gp, nodes=False)
roads_union = edges_gdf.geometry.union_all()
print(f"Corridor graph: {len(Gp.nodes):,} nodes · {len(Gp.edges):,} edges (EPSG:2039)")


In [ ]:
from shapely.geometry import Point

# Point-to-nearest-road distance: raw vs Kalman (the honest-failure metric).
resid_raw = np.array([Point(x, y).distance(roads_union) for x, y in zip(sx, sy)])
resid_kal = np.array([Point(x, y).distance(roads_union) for x, y in kal])

# (1) map: raw pings vs Kalman polyline (toggle). Transform both back to lat/lon.
inv = Transformer.from_crs("EPSG:2039", "EPSG:4326", always_xy=True)
kal_lon, kal_lat = inv.transform(kal[:, 0], kal[:, 1])
m3 = base_map([single.lat.mean(), single.lon.mean()], zoom=13)
rawfg = folium.FeatureGroup(name="raw pings")
for lat, lon in single[["lat", "lon"]].values:
    folium.CircleMarker([lat, lon], radius=3, color="#888", fill=True, fill_opacity=0.7, weight=0).add_to(rawfg)
rawfg.add_to(m3)
folium.PolyLine(list(zip(kal_lat, kal_lon)), color="#ee6677", weight=3,
                opacity=0.9, tooltip="Kalman-smoothed track").add_to(
    folium.FeatureGroup(name="Kalman track").add_to(m3))
LayerControl(collapsed=False).add_to(m3)
m3

In [ ]:
# (2) before/after histogram: point-to-nearest-road distance, raw vs Kalman.
fig, ax = plt.subplots(figsize=(6.5, 3.4))
bins = np.linspace(0, max(resid_raw.max(), resid_kal.max()), 30)
ax.hist(resid_raw, bins=bins, alpha=0.6, label=f"raw (median {np.median(resid_raw):.1f} m)", color="#888")
ax.hist(resid_kal, bins=bins, alpha=0.6, label=f"Kalman (median {np.median(resid_kal):.1f} m)", color="#ee6677")
ax.set(title="Point-to-nearest-road distance — raw vs Kalman",
       xlabel="meters", ylabel="number of pings"); ax.legend()
plt.tight_layout(); plt.show()
print("Kalman fixes jitter, but the distribution does NOT collapse onto 0 — the road is not found.")

**You can now:** apply a Kalman filter to GPS and explain why it's the wrong
tool here — and what that failure tells us we need: a **road prior**. That is
exactly the HMM's transition model.


## Step 4 — Set up HMM map matching

**Concept:** cast matching as a Hidden Markov Model. Hidden **states = candidate
road edges** near each ping; **observations = the noisy pings**; the **emission
model = the noise prior** (a Gaussian with std `obs_noise` = **emission sigma**);
the **transition model = the road network** (only on-road moves are cheap). We
build a Leuven `InMemMap` from the osmnx graph (projected, `use_latlon=False`) and
configure a `DistanceMatcher`.

**DIRECT vocabulary:** HMM (states / observations / emission / transition) ·
candidate edges · emission sigma · map matching.

*What to notice:* the matcher scores **roads**, not free space — every ping is
explained by a nearby road segment, weighted by how far it is.


In [ ]:
from leuvenmapmatching.map.inmem import InMemMap

# Build the matchable map from the projected osmnx graph. Leuven uses (y, x) = (row, col).
# Add edges in BOTH directions so the matcher can traverse the corridor either way.
def build_inmem(Gp):
    m = InMemMap("ta-13429", use_latlon=False, use_rtree=True, index_edges=True)
    nodexy = {}
    for nid, d in Gp.nodes(data=True):
        m.add_node(int(nid), (d["y"], d["x"])); nodexy[int(nid)] = (d["x"], d["y"])
    for u, v, _ in Gp.edges(keys=True):
        m.add_edge(int(u), int(v)); m.add_edge(int(v), int(u))
    m.purge()
    return m, nodexy

inmap, nodexy = build_inmem(Gp)
print(f"InMemMap built: {len(nodexy):,} nodes, bidirectional edges, rtree index on.")

In [ ]:
# (1) Candidate edges within max_dist for 3 consecutive pings — "states = candidate edges".
import geopandas as gpd
MAX_DIST = 200.0          # candidate search radius (m) — a HARD cut; we lean on the emission model
probe_idx = [len(single)//2 - 1, len(single)//2, len(single)//2 + 1]
m4 = base_map([single.lat.iloc[probe_idx[1]], single.lon.iloc[probe_idx[1]]], zoom=16)
edges_ll = edges_gdf.to_crs("EPSG:4326")
for j, pi in enumerate(probe_idx):
    px, py = sx[pi], sy[pi]
    folium.CircleMarker([single.lat.iloc[pi], single.lon.iloc[pi]], radius=6,
                        color="black", fill=True, fill_color="yellow", fill_opacity=1,
                        popup=f"ping {pi}").add_to(m4)
    # max_dist radius
    folium.Circle([single.lat.iloc[pi], single.lon.iloc[pi]], radius=MAX_DIST,
                  color="#4477aa", weight=1, fill=False).add_to(m4)
    # candidate road segments within max_dist
    near = edges_gdf[edges_gdf.distance(Point(px, py)) < MAX_DIST]
    cfg = folium.FeatureGroup(name=f"candidates · ping {pi}")
    for geom in near.to_crs("EPSG:4326").geometry:
        folium.PolyLine([(y, x) for x, y in geom.coords], color="#ee6677",
                        weight=4, opacity=0.7).add_to(cfg)
    cfg.add_to(m4)
LayerControl(collapsed=False).add_to(m4)
m4

In [ ]:
# (2) emission curve: Gaussian emission weight vs distance, for two sigmas.
d = np.linspace(0, 60, 200)
fig, ax = plt.subplots(figsize=(6, 3.2))
for sig, col in [(5, "#ee6677"), (50, "#4477aa")]:
    ax.plot(d, np.exp(-0.5 * (d / sig) ** 2), color=col, label=f"sigma = {sig} m")
ax.set(title="Emission model = the noise prior",
       xlabel="ping-to-candidate distance (m)", ylabel="emission weight (unnormalized)")
ax.legend(); plt.tight_layout(); plt.show()
print("Small sigma = strict (only very-near roads score); large sigma = forgiving (far roads still plausible).")

### (Optional, SKIPPABLE) Peek under the hood: emission × transition × Viterbi by hand

A tiny hand-rolled illustration on **3 toy candidate edges × 2 pings** — the same
math `DistanceMatcher` runs at scale. Cut this live if you are short on time; the
library does the real work in the next step.


In [ ]:
# --- SKIPPABLE toy Viterbi (no hmmlearn) ---------------------------------
sig = 15.0
# emission distances (m): ping -> each of 3 candidate edges
emit_d = np.array([[5., 25., 60.],     # ping 1: edge A closest
                   [40., 8., 50.]])     # ping 2: edge B closest
emis = np.exp(-0.5 * (emit_d / sig) ** 2)                      # Gaussian emission
# transition cost: penalize switching edges (network says staying on a road is cheap)
trans = np.array([[1.0, 0.3, 0.1],
                  [0.3, 1.0, 0.3],
                  [0.1, 0.3, 1.0]])
# Viterbi over 2 steps
V = np.zeros((2, 3)); back = np.zeros((2, 3), int)
V[0] = emis[0]
for t in (1,):
    for s in range(3):
        scores = V[t-1] * trans[:, s] * emis[t, s]
        back[t, s] = scores.argmax(); V[t, s] = scores.max()
best_last = V[1].argmax(); best = [back[1, best_last], best_last]
labels = ["A", "B", "C"]
print("emission weights per ping:\n", np.round(emis, 3))
print("Viterbi best path:", " -> ".join(labels[i] for i in best))

fig, ax = plt.subplots(figsize=(4.6, 3))
im = ax.imshow(emis.T, cmap="viridis", aspect="auto")
for t, s in enumerate(best):
    ax.scatter(t, s, s=180, facecolors="none", edgecolors="red", linewidths=2)
ax.set(xticks=[0, 1], xticklabels=["ping 1", "ping 2"], yticks=[0, 1, 2],
       yticklabels=labels, title="emission weight (red = Viterbi path)")
plt.colorbar(im, ax=ax, label="emission weight"); plt.tight_layout(); plt.show()

**You can now:** set up an HMM matcher and name what the emission model, the
transition model, and the candidate set each represent.


## Step 5 — Viterbi on the single trace

**Concept:** run `matcher.match(track)` → the **Viterbi path** as a matched edge
sequence. The road surfaces from the one sparse trace that Kalman couldn't fix.

**DIRECT vocabulary:** Viterbi algorithm · matched path · non-emitting states
(the inferred in-between positions that bridge the 1-ping/min gaps).

*What to notice:* layering raw → Kalman → **matched**, the matched route snaps
onto real roads; the point-to-road distance now collapses toward zero.


In [ ]:
from leuvenmapmatching.matcher.distance import DistanceMatcher
from shapely.geometry import LineString

def make_matcher(inmap, sigma):
    # min_prob_norm=None is load-bearing: with small sigma the normalized-probability
    # pruning otherwise kills the lattice at ping 1 (see KNOWN_ISSUES.md). max_dist is the
    # hard candidate cut; the emission model does the soft work.
    return DistanceMatcher(inmap, max_dist=MAX_DIST, max_dist_init=300,
                           obs_noise=sigma, obs_noise_ne=sigma * 2,
                           non_emitting_states=True, min_prob_norm=None,
                           max_lattice_width=50)

def match_track(inmap, sigma, lons, lats):
    "Match a lat/lon track; return (matched_nodes, matched_line_proj)."
    xs, ys = _tf.transform(lons, lats)
    mt = make_matcher(inmap, sigma)
    mt.match(list(zip(ys, xs)))                       # Leuven wants (y, x)
    nodes = mt.path_pred_onlynodes                    # PINNED accessor (smoke test)
    line = LineString([nodexy[n] for n in nodes]) if len(nodes) > 1 else None
    return nodes, line

SIGMA0 = 15.0
m_nodes, m_line = match_track(inmap, SIGMA0, single.lon.values, single.lat.values)
print(f"Matched {len(m_nodes)} road nodes · matched path length {m_line.length/1000:.1f} km")

In [ ]:
# (1) map: raw pings -> Kalman track -> matched route (toggle all three).
from folium.plugins import PolyLineTextPath
mln_lon, mln_lat = inv.transform(*np.array(m_line.coords).T)
m5 = base_map([single.lat.mean(), single.lon.mean()], zoom=13)
g_raw = folium.FeatureGroup(name="raw pings", show=True)
for lat, lon in single[["lat", "lon"]].values:
    folium.CircleMarker([lat, lon], radius=3, color="#888", fill=True, fill_opacity=0.6, weight=0).add_to(g_raw)
g_raw.add_to(m5)
folium.PolyLine(list(zip(kal_lat, kal_lon)), color="#ee6677", weight=2.5, opacity=0.8,
                tooltip="Kalman").add_to(folium.FeatureGroup(name="Kalman track", show=False).add_to(m5))
# The matched route is also DIRECTED (same travel order as Step 1): stamp arrowheads.
g_match = folium.FeatureGroup(name="HMM-matched route").add_to(m5)
ml = folium.PolyLine(list(zip(mln_lat, mln_lon)), color="#228833", weight=4, opacity=0.9,
                     tooltip="HMM-matched route")
ml.add_to(g_match)
PolyLineTextPath(ml, "  ➤  ", repeat=True, offset=3,
                 attributes={"fill": "#114422", "font-weight": "bold", "font-size": "14"}).add_to(g_match)
LayerControl(collapsed=False).add_to(m5)
print("What to notice: the matched route carries the same direction (arrowheads) as the raw "
      "trajectory in Step 1 — map matching snaps the pings to roads while preserving travel order.")
m5


In [ ]:
# (2) the quantitative spine: point-to-road distance across THREE methods.
resid_hmm = np.array([Point(x, y).distance(m_line) for x, y in zip(sx, sy)])
fig, ax = plt.subplots(figsize=(6.8, 3.4))
bins = np.linspace(0, resid_raw.max(), 30)
for arr, lab, col in [(resid_raw, "raw", "#888"),
                      (resid_kal, "Kalman", "#ee6677"),
                      (resid_hmm, "HMM-matched", "#228833")]:
    ax.hist(arr, bins=bins, alpha=0.55, label=f"{lab} (median {np.median(arr):.1f} m)", color=col)
ax.set(title="Point-to-road distance: raw vs Kalman vs HMM",
       xlabel="meters", ylabel="number of pings"); ax.legend()
plt.tight_layout(); plt.show()
print("The HMM-matched distribution collapses toward 0 — the HMM did what Kalman could not.")

**You can now:** run Viterbi on one trace and read off the matched route.


## Step 6 — Viterbi on all pooled traces: the shape crystallizes

**Concept:** match **many** runs and overlay them → the bus line's shape becomes
legible. We live-match a subset to show the mechanism, then weight each road edge
by **how many runs traversed it** (the edge-usage choropleth) for the dense reveal.

**DIRECT vocabulary:** aggregation · shape recovery · coverage.

*What to notice:* pooling **matched** (not raw) traces is what makes the shape
legible — the spine of the line lights up; occasional branches stay faint.


In [ ]:
from collections import Counter
import time

# Pure-Python Leuven is slow, so we live-match a WATCHABLE SUBSET to show the
# mechanism (~45-60 s). Set N_POOL higher (or None) to match every full run of the
# day for the dense reveal — that takes ~2 min on a fresh Colab and is the natural
# "pre-cache this and load it" artifact for a time-boxed live class.
N_POOL = 8                       # number of runs to live-match; None = all full runs
RUN_JIDS = jsize[jsize >= 30].index.tolist()
if N_POOL is not None:
    RUN_JIDS = RUN_JIDS[:N_POOL]

edge_use = Counter()
matched_lines = []
matched_nodes_by_jid = {}        # cached for step 7 (avoid re-matching to find the detour)
t0 = time.time()
for jid in RUN_JIDS:
    j = day[day.jid == jid].sort_values("recorded_at_time")
    try:
        nodes, line = match_track(inmap, SIGMA0, j.lon.values, j.lat.values)
    except Exception:
        continue
    if line is None:
        continue
    matched_lines.append((jid, line))
    matched_nodes_by_jid[jid] = nodes
    for a, b in zip(nodes[:-1], nodes[1:]):
        edge_use[frozenset((a, b))] += 1
print(f"Matched {len(matched_lines)}/{len(RUN_JIDS)} runs in {time.time()-t0:.0f}s · "
      f"{len(edge_use)} distinct edges traversed")

In [ ]:
# (1) MONEY SHOT side-by-side: raw pooled pings (step 2) vs pooled MATCHED routes.
import branca
m6 = base_map(DAY_CENTER, zoom=12)
g_rawpool = folium.FeatureGroup(name="raw pooled pings", show=False)
for lat, lon in day[["lat", "lon"]].values:
    folium.CircleMarker([lat, lon], radius=1.2, color="#bbb", fill=True, fill_opacity=0.4, weight=0).add_to(g_rawpool)
g_rawpool.add_to(m6)
g_matched = folium.FeatureGroup(name="pooled MATCHED routes")
for jid, line in matched_lines:
    lo, la = inv.transform(*np.array(line.coords).T)
    folium.PolyLine(list(zip(la, lo)), color="#228833", weight=2, opacity=0.25).add_to(g_matched)
g_matched.add_to(m6)
LayerControl(collapsed=False).add_to(m6)
m6

In [ ]:
# (2) edge-usage choropleth: roads weighted/coloured by traversal count (mapclassify bins).
import mapclassify, branca.colormap as bcm
# Map each used edge (node pair) back to its geometry via the projected edges gdf.
node_pt = {int(nid): Point(d["x"], d["y"]) for nid, d in Gp.nodes(data=True)}
counts = np.array(list(edge_use.values()))
cmax = counts.max()
cm = bcm.linear.YlOrRd_09.scale(1, cmax); cm.caption = "runs traversing this edge"
m6b = base_map(DAY_CENTER, zoom=12)
for pair, c in edge_use.items():
    a, b = tuple(pair)
    if a not in node_pt or b not in node_pt:
        continue
    (lo, la) = inv.transform([node_pt[a].x, node_pt[b].x], [node_pt[a].y, node_pt[b].y])
    folium.PolyLine(list(zip(la, lo)), color=cm(c), weight=1 + 6 * c / cmax,
                    opacity=0.85, tooltip=f"{c} runs").add_to(m6b)
cm.add_to(m6b); LayerControl(collapsed=False).add_to(m6b)
m6b

In [ ]:
# (3) edge-traversal-count histogram — the metric behind the choropleth.
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(counts, bins=range(1, cmax + 2), color="#cc6677", align="left")
ax.set(title="Edge traversal counts (pooled matched runs)",
       xlabel="number of runs traversing an edge", ylabel="number of edges")
plt.tight_layout(); plt.show()
print(f"Spine edges (traversed by many runs) = the line's shape; rare edges = detours/noise.")

**You can now:** aggregate many matched traces to recover a route's shape,
and explain why pooling *matched* (not raw) traces is what makes it legible.


## Step 7 — Interpret: a detour, a wrong match, and the sigma sweep

**Concept:** (a) spot a **detour** run that diverges from the spine; (b) find a
spot the matcher got **wrong** and reason about why; (c) **sweep the emission
sigma** and watch the matched path tighten/loosen — the unit's clearest "noise is
a modeling decision" beat.

**DIRECT vocabulary:** emission sigma sweep · sensitivity · failure mode · detour.

*What to notice:* small sigma snaps tight (and can snap *wrong*); large sigma
wanders and is more forgiving of GPS error.


In [ ]:
# (a) detour: the run whose matched path shares the FEWEST edges with the popular spine.
spine = {pair for pair, c in edge_use.items() if c >= max(2, int(0.3 * cmax))}
def overlap(line_nodes):
    pairs = {frozenset((a, b)) for a, b in zip(line_nodes[:-1], line_nodes[1:])}
    return len(pairs & spine) / max(1, len(pairs))
# reuse the node lists cached in step 6 (no re-matching) to find the most-divergent run
scored = []
for jid, line in matched_lines:
    nodes = matched_nodes_by_jid[jid]
    scored.append((overlap(nodes), jid, line))
scored.sort()
detour_jid, detour_line = scored[0][1], scored[0][2]
typical_line = scored[-1][2]
m7 = base_map(DAY_CENTER, zoom=12)
for line, name, col in [(typical_line, "typical run", "#228833"), (detour_line, "DETOUR run", "#aa3377")]:
    lo, la = inv.transform(*np.array(line.coords).T)
    folium.PolyLine(list(zip(la, lo)), color=col, weight=4, opacity=0.85,
                    tooltip=name).add_to(folium.FeatureGroup(name=name).add_to(m7))
LayerControl(collapsed=False).add_to(m7)
print(f"Most divergent run: {detour_jid} (spine overlap {scored[0][0]:.0%})")
m7

In [ ]:
# (c) SIGMA SWEEP — paired: a curve (sigma -> mean residual & path length) + toggleable maps.
SIGMAS = [5, 15, 50]
sweep = {}
for sig in SIGMAS:
    nodes, line = match_track(inmap, float(sig), single.lon.values, single.lat.values)
    res = np.array([Point(x, y).distance(line) for x, y in zip(sx, sy)])
    sweep[sig] = dict(line=line, mean_res=res.mean(), length=line.length / 1000)

fig, ax1 = plt.subplots(figsize=(6.2, 3.4))
ax2 = ax1.twinx()
ax1.plot(SIGMAS, [sweep[s]["mean_res"] for s in SIGMAS], "o-", color="#4477aa", label="mean residual")
ax2.plot(SIGMAS, [sweep[s]["length"] for s in SIGMAS], "s--", color="#ee6677", label="matched length")
ax1.set(xlabel="emission sigma (m)", ylabel="mean ping-to-road residual (m)")
ax2.set_ylabel("matched-path length (km)")
ax1.set_title("Sigma sweep: noise is a modeling decision")
fig.legend(loc="upper center", ncol=2); plt.tight_layout(); plt.show()
for s in SIGMAS:
    print(f"sigma={s:3d} m  mean residual {sweep[s]['mean_res']:5.1f} m  length {sweep[s]['length']:.1f} km")

In [ ]:
# the maps half of the pair: per-sigma matched paths as toggleable layers on ONE map.
m7b = base_map([single.lat.mean(), single.lon.mean()], zoom=13)
g0 = folium.FeatureGroup(name="raw pings", show=False)
for lat, lon in single[["lat", "lon"]].values:
    folium.CircleMarker([lat, lon], radius=2.5, color="#888", fill=True, fill_opacity=0.5, weight=0).add_to(g0)
g0.add_to(m7b)
colors = {5: "#ee6677", 15: "#228833", 50: "#4477aa"}
for sig in SIGMAS:
    lo, la = inv.transform(*np.array(sweep[sig]["line"].coords).T)
    folium.PolyLine(list(zip(la, lo)), color=colors[sig], weight=3.5, opacity=0.85,
                    tooltip=f"sigma = {sig} m").add_to(
        folium.FeatureGroup(name=f"matched · sigma={sig} m", show=(sig == 15)).add_to(m7b))
LayerControl(collapsed=False).add_to(m7b)
m7b

**You can now:** tune emission sigma and interpret matcher output — name why
a match failed and predict how sigma changes it. (Newson & Krumm estimate
sigma ~ 4 m for car GPS; buses in an urban canyon often want a bit more.)


## Step 8 (Advanced, ~5 min — *trimmable*) — Bearing + velocity-aware matching

**Concept:** the base matcher uses only *where* each ping is (its distance to a
road). But the SIRI feed also reports **`bearing`** (which way the bus is facing)
and **`velocity`** — two extra observation channels the emission model ignored.
On a **divided road** the two carriageways sit a few meters apart and point in
**opposite** directions; distance alone can snap a ping onto the *wrong*
carriageway (exactly the failure mode Step 7 surfaced). Here we add two
interpretable terms — **no torch, no new heavy deps**, just an extended
emission/transition:

1. **Heading agreement** reweights the *emission*: a candidate edge whose road
   direction disagrees with the ping's reported `bearing` is penalized
   (cos-similarity of the two headings; an opposite carriageway is maximally
   penalized).
2. **A velocity gate** penalizes a between-ping *transition* that would require an
   implausible speed — farther than `(reported velocity + slack) × time-gap`.

> *Learned matchers (L2MM/DiffMM) are the frontier here, but there is no
> Colab-friendly pretrained checkpoint (June 2026), so that comparison stays
> slide-only.* This classical version is fully transparent — you can read every term.


In [ ]:
# Heading + velocity terms, computed transparently for the demo. Compass bearing
# of a directed road segment (0=N, 90=E) from its projected (x,y) endpoints:
def edge_bearing_xy(x1, y1, x2, y2):
    return np.degrees(np.arctan2(x2 - x1, y2 - y1)) % 360   # atan2(dE, dN)

def heading_agreement(ping_bearing, edge_brg):
    "cos similarity of two compass headings: +1 aligned, 0 perpendicular, -1 opposite."
    d = abs((edge_brg - ping_bearing + 180) % 360 - 180)
    return np.cos(np.radians(d))

# --- Intuition figure (REQUIRED): one ping, its candidate edges, heading agreement ---
# Pick the carriageway-ambiguity spot Step 7's failure mode points at: a divided road
# where the bus's reported heading disambiguates which side it is on.
PI = 65                                              # ping index on a divided road (Jabotinsky)
pb = float(single.bearing.iloc[PI]); pv = float(single.velocity.iloc[PI])
ppx, ppy = sx[PI], sy[PI]
cand = edges_gdf[edges_gdf.distance(Point(ppx, ppy)) < 80].copy()
rows = []
for _, r in cand.iterrows():
    c = list(r.geometry.coords)
    eb = edge_bearing_xy(c[0][0], c[0][1], c[-1][0], c[-1][1])
    rows.append((eb, heading_agreement(pb, eb), str(r.get("name"))[:22]))
rows.sort(key=lambda t: -t[1])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
# left: candidate roads near the ping, coloured by heading agreement
agree_vals = [a for _, a, _ in rows]
sc = None
for _, r in cand.iterrows():
    c = list(r.geometry.coords)
    eb = edge_bearing_xy(c[0][0], c[0][1], c[-1][0], c[-1][1])
    ag = heading_agreement(pb, eb)
    xs_, ys_ = zip(*[(pt[0], pt[1]) for pt in c])
    ax[0].plot(xs_, ys_, color=plt.cm.RdYlGn((ag + 1) / 2), lw=3, solid_capstyle="round")
ax[0].plot(ppx, ppy, "o", ms=11, mfc="yellow", mec="black", zorder=5)
# draw the reported heading as an arrow from the ping
arr = 60
ax[0].annotate("", xy=(ppx + arr*np.sin(np.radians(pb)), ppy + arr*np.cos(np.radians(pb))),
               xytext=(ppx, ppy), arrowprops=dict(arrowstyle="-|>", color="black", lw=2))
ax[0].set(title=f"Candidate roads near ping {PI}\n(green=agrees with reported heading {pb:.0f}°, red=opposite)",
          xlabel="easting (m)", ylabel="northing (m)"); ax[0].set_aspect("equal"); ax[0].axis("off")
# right: the velocity gate as a feasibility curve
gap_demo = 60.0                                       # a 60-s between-ping gap
vmps = pv / 3.6
slack = 8.0
reach = (vmps + slack) * gap_demo
dists = np.linspace(0, reach * 1.8, 200)
penalty = np.maximum(0.0, dists / gap_demo - (vmps + slack))
ax[1].plot(dists, penalty, color="#cc6677", lw=2)
ax[1].axvline(reach, color="k", ls="--", lw=1)
ax[1].text(reach*1.02, penalty.max()*0.6, f"plausible reach\n({pv:.0f} km/h + slack, {gap_demo:.0f}s)", fontsize=8)
ax[1].set(title="Velocity gate: penalty vs between-ping move distance",
          xlabel="transition distance (m)", ylabel="speed-excess penalty (m/s)")
plt.tight_layout(); plt.show()
print(f"Ping {PI}: reported heading {pb:.0f}°, velocity {pv:.0f} km/h. "
      f"Best-agreeing candidate {rows[0][2]} ({rows[0][0]:.0f}°, agree {rows[0][1]:+.2f}); "
      f"worst {rows[-1][2]} ({rows[-1][0]:.0f}°, agree {rows[-1][1]:+.2f}).")


In [ ]:
# Direction-aware matcher: subclass DistanceMatcher and add the two terms to the
# emission (logprob_obs) and transition (logprob_trans). Both terms are <= 0
# (log-probs), so they only ever PENALIZE — never push a log-prob above 0
# (Leuven asserts logprob_obs <= 0). The observation index is recovered from the
# edge_o label ("O<idx>") that Leuven assigns during matching.
class DirAwareMatcher(DistanceMatcher):
    HEAD_W = 12.0      # heading-disagreement weight (log-prob units)
    VEL_W  = 4.0       # velocity-gate weight
    VEL_SLACK = 8.0    # m/s headroom over reported speed (~29 km/h) before penalizing

    def configure_dir(self, bearings, speeds_mps, gaps_s):
        self._brg, self._spd, self._gap = bearings, speeds_mps, gaps_s

    def logprob_obs(self, dist, prev_m=None, new_edge_m=None, new_edge_o=None, is_ne=False):
        lp, props = super().logprob_obs(dist, prev_m, new_edge_m, new_edge_o, is_ne)
        try:
            b = self._brg[int(new_edge_o.label[1:])]
        except Exception:
            return lp, props
        if b is None or new_edge_m is None or new_edge_m.is_point():
            return lp, props
        eb = edge_bearing_xy(new_edge_m.p1[1], new_edge_m.p1[0],
                             new_edge_m.p2[1], new_edge_m.p2[0])   # Leuven points are (y,x)
        agree = heading_agreement(b, eb)
        return lp + self.HEAD_W * (agree - 1.0), props             # 0 aligned … -2·W opposite

    def logprob_trans(self, prev_m, edge_m, edge_o, is_prev_ne=False, is_next_ne=False):
        lp, props = super().logprob_trans(prev_m, edge_m, edge_o, is_prev_ne, is_next_ne)
        try:
            oi = int(edge_o.label[1:]); gap = self._gap[oi]; v = self._spd[oi]
        except Exception:
            return lp, props
        if gap is None or gap <= 0 or edge_m.is_point():
            return lp, props
        step = np.hypot(edge_m.p2[0]-edge_m.p1[0], edge_m.p2[1]-edge_m.p1[1])
        excess = max(0.0, step/gap - (v + self.VEL_SLACK))         # implausible-speed excess
        return lp - self.VEL_W * excess, props

def match_dir(inmap, sigma, j):
    "Direction-aware match of one journey; returns (nodes, projected line)."
    xs, ys = _tf.transform(j.lon.values, j.lat.values)
    mt = DirAwareMatcher(inmap, max_dist=MAX_DIST, max_dist_init=300,
                         obs_noise=sigma, obs_noise_ne=sigma*2, non_emitting_states=True,
                         min_prob_norm=None, max_lattice_width=50)
    mt.configure_dir(list(j.bearing.values),
                     list(j.velocity.values / 3.6),                # km/h -> m/s
                     list(j.recorded_at_time.diff().dt.total_seconds().values))
    mt.match(list(zip(ys, xs)))
    nodes = mt.path_pred_onlynodes
    return nodes, (LineString([nodexy[n] for n in nodes]) if len(nodes) > 1 else None)

base_nodes, base_line = match_track(inmap, SIGMA0, single.lon.values, single.lat.values)
dir_nodes,  dir_line  = match_dir(inmap, SIGMA0, single)
changed = set(base_nodes) ^ set(dir_nodes)
print(f"Base match: {len(base_nodes)} nodes · direction-aware: {len(dir_nodes)} nodes · "
      f"{len(changed)} nodes differ — the heading/velocity terms changed the route.")


In [ ]:
# RESULT (before/after): the wrong-carriageway spot, base vs direction-aware (toggle).
# Centre on the divided-road ping; show both matched routes + the reported-heading arrow.
ctr = [single.lat.iloc[PI], single.lon.iloc[PI]]
m8 = base_map(ctr, zoom=16)
# raw pings (faint) for context
g_raw = folium.FeatureGroup(name="raw pings", show=False)
for lat, lon in single[["lat", "lon"]].values:
    folium.CircleMarker([lat, lon], radius=2.5, color="#888", fill=True, fill_opacity=0.5, weight=0).add_to(g_raw)
g_raw.add_to(m8)
# the focus ping + its reported heading arrow (a short directed marker)
hb = np.radians(pb)
tip = (ctr[0] + 0.0006*np.cos(hb), ctr[1] + 0.0006*np.sin(hb)/np.cos(np.radians(ctr[0])))
folium.PolyLine([ctr, list(tip)], color="black", weight=3,
                tooltip=f"reported heading {pb:.0f}°").add_to(m8)
folium.CircleMarker(ctr, radius=6, color="black", fill=True, fill_color="yellow",
                    fill_opacity=1, popup=f"ping {PI}: heading {pb:.0f}°, {pv:.0f} km/h").add_to(m8)
for line, name, col, sh in [(base_line, "base match (distance only)", "#ee6677", True),
                            (dir_line, "direction-aware (heading+velocity)", "#228833", True)]:
    lo, la = inv.transform(*np.array(line.coords).T)
    folium.PolyLine(list(zip(la, lo)), color=col, weight=4, opacity=0.85,
                    tooltip=name).add_to(folium.FeatureGroup(name=name, show=sh).add_to(m8))
LayerControl(collapsed=False).add_to(m8)
m8


*What to notice:* zoom to the black arrow (the bus's reported heading). The
**distance-only** match can sit on the *opposite* carriageway of the divided road;
the **direction-aware** match snaps to the carriageway whose road direction agrees
with the reported bearing. A second observation channel resolves an ambiguity that
emission **sigma alone cannot** — sigma controls *how forgiving* the noise model
is, not *which direction* the bus faces.

**You can now:** name two extra observation channels (heading, speed), say how each
enters the HMM (heading → emission reweight; speed → transition gate), and predict
when they matter (divided roads, parallel carriageways, dense junctions).


## Step 9 — The inverse problem: infer the road graph from pings alone

**Concept:** flip the inference direction. Steps 4–8 ran the noise frame
**forward**: given the road graph *and* the pings, recover the trajectory. Now run
it **backward** — **no OSM to match to** — and recover the *graph itself* from the
pings. This is **map inference**, the spine of the supervised practice.

The classic density→skeleton pipeline (Biagioni & Eriksson 2012):
**KDE density of the pooled pings → threshold to a binary raster →
`skimage.skeletonize(method="lee")` → trace it to a small `networkx` graph →
overlay (buffered) against OSM** to see what we recovered. The KDE **bandwidth**
is the inverse-problem analogue of the **emission sigma** you swept in Step 7: too
small and the centreline frays into noise; too large and parallel roads merge.

We also name **DTW** (the practice's shape-comparison tool) and bridge to **Unit 3**:
matched per-segment traversals become a **segment-speed time series** (Lagrangian →
Eulerian).

**DIRECT vocabulary:** map inference / inverse problem · KDE bandwidth (≈ emission
sigma) · skeletonization · DTW · segment-speed series.

> This is a **preview**: we do density→skeleton→graph→overlay. The full
> **buffered precision/recall evaluation** of inferred-vs-OSM is the practice
> (handed off below in *Where to go next*).

> **Scaling to the practice (heads-up).** Here we call `scipy.stats.gaussian_kde` on **one line**, and its `bw_method` (e.g. `0.02`) is a *dimensionless* scaling — fine for a quick single-line preview. The supervised practice runs this **city-wide**, where a per-point `gaussian_kde` is too slow, so it re-expresses the *same idea* as a **raster**: bin the pings into ~30 m cells and **Gaussian-blur with a bandwidth measured in metres**. That makes the “bandwidth ≈ emission σ” analogy literal (both in metres) and the grade fast (dilate two boolean rasters, count overlap). The reference solution's `infer_graph` shows that raster form — it, not this KDE call, is the practice's starting block.


In [ ]:
# (1) inverse-problem schematic: pings -> (no OSM) -> inferred graph (two panels).
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].scatter(day.lon, day.lat, s=1, color="#888", alpha=0.3)
axs[0].set(title="FORWARD (this demo): pings + OSM -> trajectory", xlabel="lon", ylabel="lat")
# crude density to hint at the inferred-graph spine for the schematic
axs[1].hexbin(day.lon, day.lat, gridsize=60, cmap="YlOrRd", mincnt=1)
axs[1].set(title="INVERSE (the practice): pings ALONE -> graph", xlabel="lon")
plt.tight_layout(); plt.show()

In [ ]:
# (B1) DENSITY: KDE of the pooled day's pings, projected to EPSG:2039 (meters).
# Bandwidth here is the inverse-problem analogue of the emission sigma from Step 7.
from scipy.stats import gaussian_kde
dx, dy = _tf.transform(day.lon.values, day.lat.values)
xmin, xmax, ymin, ymax = dx.min(), dx.max(), dy.min(), dy.max()
# SQUARE PIXELS: anchor the grid on a fixed METRIC cell size (not a fixed square
# count). A fixed GRID over the corridor's non-square bbox gave ~45 m x 62 m
# pixels (ratio 1.39), which distorts skeletonize and pushes inferred vertices
# off-road. CELL_M meters/pixel keeps pixels ~square.
CELL_M = 30
nx_g = max(2, round((xmax - xmin) / CELL_M))     # cols = easting (x)
ny_g = max(2, round((ymax - ymin) / CELL_M))     # rows = northing (y)
gx = np.linspace(xmin, xmax, nx_g)               # length nx_g, indexes COLUMNS (easting)
gy = np.linspace(ymin, ymax, ny_g)               # length ny_g, indexes ROWS (northing)
XX, YY = np.meshgrid(gx, gy)                      # shape (ny_g, nx_g)
KDE_BW = 0.02                                     # <-- the "emission sigma" of map inference (less smoothing)
kde = gaussian_kde(np.vstack([dx, dy]), bw_method=KDE_BW)
# reshape (ny_g, nx_g): rows = y/northing (len ny_g), cols = x/easting (len nx_g).
dens = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(ny_g, nx_g)

fig, ax = plt.subplots(figsize=(5.6, 5.2))
im = ax.imshow(dens, origin="lower", extent=[xmin, xmax, ymin, ymax],
               cmap="magma", aspect="equal")
ax.set(title=f"Ping density (KDE bw={KDE_BW}, {CELL_M} m/px, grid {ny_g}x{nx_g})",
       xlabel="easting (m)", ylabel="northing (m)")
fig.colorbar(im, ax=ax, shrink=0.8, label="density"); plt.tight_layout(); plt.show()
print(f"Grid {ny_g} rows x {nx_g} cols at {CELL_M} m/px "
      f"(pixel {(xmax-xmin)/(nx_g-1):.0f} m x {(ymax-ymin)/(ny_g-1):.0f} m).")
print("What to notice: the corridor's spine glows — but it is a fuzzy blob, not yet a graph.")


In [ ]:
# (B2) SKELETON: threshold the density, then skimage.skeletonize -> a 1-pixel centreline.
from skimage.morphology import skeletonize
THR_Q = 0.88
thr = np.quantile(dens[dens > 0], THR_Q)
mask = dens >= thr                                 # bool raster (skeletonize needs bool), shape (ny_g, nx_g)
skel = skeletonize(mask, method="lee")             # 'lee' = robust on thick blobs; keeps (ny_g, nx_g)
assert skel.dtype == bool

fig, ax = plt.subplots(1, 2, figsize=(10.5, 5))
ax[0].imshow(mask, origin="lower", extent=[xmin, xmax, ymin, ymax], cmap="Greys", aspect="equal")
ax[0].set(title=f"Thresholded density (q={THR_Q}) — the blob", xlabel="easting (m)", ylabel="northing (m)")
ax[1].imshow(mask, origin="lower", extent=[xmin, xmax, ymin, ymax], cmap="Greys", alpha=0.25, aspect="equal")
# np.nonzero on a (ny_g, nx_g) array -> (rows, cols) = (northing idx, easting idx).
rows_i, cols_i = np.nonzero(skel)
ax[1].scatter(gx[cols_i], gy[rows_i], s=1.5, color="#cc3311")   # gx indexed by cols, gy by rows
ax[1].set(title=f"Skeleton (1-px centreline, {int(skel.sum())} px)", xlabel="easting (m)")
plt.tight_layout(); plt.show()
print("What to notice: skeletonize thins the blob to a 1-pixel ridge — the candidate road centreline.")


In [ ]:
# (B3) TRACE the skeleton into a compact networkx graph (junctions/endpoints + chains).
# Optional `sknw` does this in one call; we fall back to a transparent 8-neighbour
# pixel-graph + degree-2 chain contraction so the cell runs with no extra dependency.
import networkx as nx
# px_to_world: c indexes easting (gx), r indexes northing (gy). skel is (ny_g, nx_g).
def px_to_world(r, c): return gx[c], gy[r]
try:
    import sknw                                    # optional one-liner skeleton->graph
    Gsk = sknw.build_sknw(skel.astype(np.uint16))
    infer_edges = []
    for u, v, d in Gsk.edges(data=True):
        pts = [px_to_world(int(r), int(c)) for r, c in d["pts"]]
        infer_edges.append(pts)
    print(f"[sknw] inferred graph: {Gsk.number_of_nodes()} nodes · {len(infer_edges)} edges")
except Exception:
    pix = set(zip(*[a.tolist() for a in np.nonzero(skel)]))   # (row,col)
    Gpx = nx.Graph()
    for (r, c) in pix:
        Gpx.add_node((r, c))
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if (dr or dc) and (r+dr, c+dc) in pix:
                    Gpx.add_edge((r, c), (r+dr, c+dc))
    deg = dict(Gpx.degree())
    keep = {n for n in Gpx if deg[n] != 2}          # junctions + endpoints
    def walk(start, nb):
        path = [start, nb]; prev, cur = start, nb
        while deg.get(cur, 0) == 2:
            nxt = [x for x in Gpx[cur] if x != prev]
            if not nxt: break
            prev, cur = cur, nxt[0]; path.append(cur)
        return path
    seen, infer_edges = set(), []
    for n in keep:
        for nb in Gpx[n]:
            path = walk(n, nb); end = path[-1]
            if end in keep and frozenset((n, end)) not in seen:
                seen.add(frozenset((n, end)))
                infer_edges.append([px_to_world(r, c) for r, c in path])
    print(f"[fallback] inferred graph: {len(keep)} junction/endpoint nodes · {len(infer_edges)} edges")
infer_km = sum(np.hypot(np.diff([p[0] for p in e]), np.diff([p[1] for p in e])).sum()
               for e in infer_edges) / 1000
print(f"Inferred network length: {infer_km:.1f} km  (no OSM was used to produce it)")

# DIAGNOSTIC: how far do inferred vertices sit from the nearest OSM road?
# Both infer_edges and roads_union are in EPSG:2039 (meters), so distances are metric.
from shapely.geometry import Point as _Pt
_verts = np.array([pt for e in infer_edges for pt in e])
_d2road = np.array([_Pt(x, y).distance(roads_union) for x, y in _verts])
print(f"Inferred-vertex -> nearest OSM road (m):  "
      f"median {np.median(_d2road):.0f}  ·  p90 {np.percentile(_d2road, 90):.0f}  "
      f"·  max {_d2road.max():.0f}   (n={len(_verts)})")


In [ ]:
# (B4) OVERLAY the INFERRED graph on the OSM TILE basemap (ground truth).
# We DO NOT draw all ~43,837 OSM edges as polylines: that bloats the HTML so much
# Colab refuses to render the map. The OSM/Carto tile layer already shows the real
# road network; we overlay only the inferred centrelines, coloured by how far each
# edge sits from the nearest OSM road (green = on-road, red = wandering).
import branca.colormap as bcm
m9 = base_map(DAY_CENTER, zoom=12)

# light context: a SAMPLE of the pooled pings (not all of them) so students see
# the evidence the inference was built from, without bloating the map.
samp = day.sample(min(1500, len(day)), random_state=0)
g_ctx = folium.FeatureGroup(name="pooled pings (sample, context)", show=False)
for lat, lon in samp[["lat", "lon"]].values:
    folium.CircleMarker([lat, lon], radius=1, color="#888", fill=True,
                        fill_opacity=0.4, weight=0).add_to(g_ctx)
g_ctx.add_to(m9)

# per-edge mean off-road distance (reuse roads_union from Step 3; metric EPSG:2039)
from shapely.geometry import Point as _Pt
edge_off = [float(np.mean([_Pt(x, y).distance(roads_union) for x, y in e])) for e in infer_edges]
cmap9 = bcm.linear.RdYlGn_09.scale(0, 60).to_step(n=6)   # 0 m green (on-road) -> 60 m+ red
cmap9.caption = "inferred edge: mean dist to nearest OSM road (m)"

g_inf = folium.FeatureGroup(name="INFERRED graph (from pings only)", show=True)
for e, off in zip(infer_edges, edge_off):
    exs = [pt[0] for pt in e]; eys = [pt[1] for pt in e]
    lo, la = inv.transform(exs, eys)
    folium.PolyLine(list(zip(la, lo)), color=cmap9(min(off, 60)), weight=3,
                    opacity=0.9, tooltip=f"mean off-road {off:.0f} m").add_to(g_inf)
g_inf.add_to(m9)
cmap9.add_to(m9); LayerControl(collapsed=False).add_to(m9)
print("What to notice: the inferred centrelines (red->green by off-road distance) recover the "
      "bus corridor's spine WITHOUT OSM, drawn over the OSM tile basemap (the ground truth).")
print("Where they miss OSM roads = no buses ran there (coverage), not a graph bug. "
      "Quantifying that gap (buffered precision/recall) is the practice.")
m9


In [ ]:
# (2) DTW teaser: align two matched runs of variable speed (tslearn).
from tslearn.metrics import dtw, dtw_path
# cumulative distance vs ping index for two full runs (proxy for along-route progress).
def cumdist(jid):
    j = day[day.jid == jid].sort_values("recorded_at_time")
    xs, ys = _tf.transform(j.lon.values, j.lat.values)
    return np.concatenate([[0], np.cumsum(np.hypot(np.diff(xs), np.diff(ys)))]) / 1000
two = [jid for jid, _ in matched_lines][:2]
a, b = cumdist(two[0]), cumdist(two[1])
path_align, dist = dtw_path(a, b)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(a, label="run A"); ax[0].plot(b, label="run B")
ax[0].set(title="Along-route progress (variable speed)", xlabel="ping #", ylabel="km"); ax[0].legend()
ax[1].plot([p[0] for p in path_align], [p[1] for p in path_align], color="#cc6677")
ax[1].set(title=f"DTW warping path (DTW dist = {dist:.2f})", xlabel="run A index", ylabel="run B index")
plt.tight_layout(); plt.show()
print("DTW aligns the two runs despite different speeds/dwells — Euclidean (index-by-index) cannot.")

In [ ]:
# (3) bridge to U3: segment-speed time series for one matched edge (Lagrangian -> Eulerian).
# Use the busiest journey's reported velocities as a stand-in segment-speed series.
sp = single.copy()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(sp.recorded_at_time, sp.velocity, "-o", ms=3, color="#4477aa")
ax.set(title="One run's reported speed over time (a segment-speed series feeds U3)",
       xlabel="time (UTC)", ylabel="km/h"); fig.autofmt_xdate()
plt.tight_layout(); plt.show()
print("Matched per-segment traversals -> speed time series at fixed road segments = Unit 3's Eulerian view.")

## Where to go next

Four open explorations that match the supervised-practice framing:

1. **Stress the matcher's recall.** Swap `line_ref=13429` for a **peripheral
   line** (regenerate the artifact via the optional cell, or change `DRIVE_ID`),
   or shrink the corridor bbox, and watch where matched recall drops. *Which roads
   does the matcher miss, and why?*
2. **Tune the direction terms.** Step 8 added heading + velocity with fixed weights
   (`HEAD_W`, `VEL_W`, `VEL_SLACK`). Sweep them: how strong must the heading term be
   before it flips a carriageway — and when does it *over*-correct on a genuine
   U-turn or loop? *Where does a second observation channel help vs. hurt?*
3. **Score the inferred graph (the practice).** Step 9 *previewed* map inference
   (KDE → skeletonize → `networkx`). Now **evaluate** it: buffer the inferred and
   OSM edges and compute **precision/recall**; sweep the **KDE bandwidth** (the
   emission-sigma analogue) and the density threshold. *Where does ping coverage,
   not the algorithm, limit recall — and how do parallel roads merge as bandwidth
   grows?*
4. **Feed the matches downstream (Unit 3).** Turn matched per-segment traversals
   into a **segment-speed time series** and start asking Eulerian questions: when
   is this corridor congested, and how predictable is it?


---
## (OPTIONAL — NOT run in class) Provenance: regenerate the line artifact from the raw archive

This is the exact, reproducible recipe that produced `ta_line_13429.parquet` from
the **canonical raw Drive archive** — so the data is self-contained from source,
not a hand-staged mystery file. It is **import-guarded and not on the class
runtime path**: it pulls a 1.2 GB ZIP whose only useful member is a 730 MB parquet
(the rest is an irrelevant SQL dump). Run it only if you want to re-derive the cut
or pick a different line.


In [ ]:
REGENERATE = False   # set True to re-derive the artifact from the raw archive (~1.2 GB pull)
if REGENERATE:
    import gdown, zipfile
    import pyarrow.parquet as pq, pyarrow.compute as pc
    ARCHIVE_ID = "1BniXRYr5DrYvY_miHpD-hjChFB8s1ac6"          # full raw archive (ZIP)
    gdown.download(id=ARCHIVE_ID, output="busarchive.bin", quiet=False)
    z = zipfile.ZipFile("busarchive.bin")
    with z.open("exports/vehicle_locations.parquet") as s, open("vl.parquet", "wb") as d:
        d.write(s.read())                                     # extract ONLY this member (skip 2.35 GB SQL)
    cols = ["recorded_at_time", "lon", "lat", "bearing", "velocity", "line_ref", "vehicle_ref"]
    full = pq.ParquetFile("vl.parquet").read(columns=cols)    # column projection
    line = full.filter(pc.equal(full["line_ref"], "13429"))   # TA line -> ~0.72 MB
    pq.write_table(line, "ta_line_13429.parquet", compression="zstd")
    print("Regenerated ta_line_13429.parquet from the raw archive.")
else:
    print("Provenance cell is inert (REGENERATE=False). The class path uses the small Drive artifact.")